# Live Demo 2 — Optimising the fine-tuned detector for CPU inference

We take the Faster R-CNN (MobileNetV3-FPN) fine-tuned in `01_object_detection_pennfudan.ipynb` and ask:
**what actually makes it faster on a CPU, and what does each trick cost in accuracy?**

Every variant is measured the same way:
* **Latency** — per-image wall-clock time at batch 1 (as in an online service): warm-up on every benchmark image,
  then 3 trials × 24 timed calls on 12 held-out images; we report the **best trial's median** (like Python's
  `timeit`), because background activity on a laptop can only make a trial slower.
* **Size** — serialised weights (`state_dict` or `.onnx` file).
* **AP@0.5** — on the 50 held-out test images (same metric code as notebook 01).

| # | Knob | Changes weights? | Section |
|---|---|---|---|
| a | baseline FP32 eager, 800 px, 1000 proposals, all threads | — | (a) |
| b | input resolution 800 → 640 → 512 → 416 → 320 | no | (b) |
| c | PyTorch dynamic INT8 quantization (Linear layers) | yes | (c) |
| d | fewer RPN proposals (1000 → 300 → 100 → 50) | no | (d) |
| e | number of CPU threads | no | (e) |
| f | ONNX export + ONNX Runtime | no (same maths) | (f) |
| g | best combination | | (g) |

> **Trainer note:** needs `checkpoints/fasterrcnn_mbv3_pennfudan.pth` from notebook 01. Whole notebook ≈ 4–5 min on
> the reference laptop (see README). Absolute numbers depend on the laptop and on background load — the **ratios** are
> the lesson. Close other heavy apps before running it live.

## 0. Setup: data split, metric and the fine-tuned model (same code as notebook 01)

In [ ]:
from pathlib import Path
import time, copy, io, warnings
import numpy as np
import pandas as pd
import torch, torchvision
from torch.utils.data import Dataset, Subset
from torchvision.ops import box_iou
import torchvision.transforms.functional as TF
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn,
                                          FasterRCNN_MobileNet_V3_Large_FPN_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
DATA_DIR = Path("..") / "data" / "PennFudanPed"
CKPT_DIR, OUT_DIR = Path("checkpoints"), Path("outputs")
OUT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "fasterrcnn_mbv3_pennfudan.pth"
assert CKPT_PATH.exists(), "Run 01_object_detection_pennfudan.ipynb first (it creates the checkpoint)"
MAX_THREADS = torch.get_num_threads()
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | default threads = {MAX_THREADS}")

class PennFudanDataset(Dataset):
    def __init__(self, root):
        self.root = Path(root)
        self.imgs = sorted((self.root / "PNGImages").glob("*.png"))
        self.masks = sorted((self.root / "PedMasks").glob("*.png"))
        self.boxes = [self._boxes(m) for m in self.masks]
    @staticmethod
    def _boxes(p):
        mask = np.array(Image.open(p)); ids = np.unique(mask); ids = ids[ids != 0]; out = []
        for i in ids:
            ys, xs = np.where(mask == i)
            x1, y1, x2, y2 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
            if x2 - x1 > 1 and y2 - y1 > 1: out.append([x1, y1, x2, y2])
        return torch.tensor(out, dtype=torch.float32).reshape(-1, 4)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = TF.pil_to_tensor(Image.open(self.imgs[i]).convert("RGB")).float() / 255.0
        return img, {"boxes": self.boxes[i].clone(), "labels": torch.ones(len(self.boxes[i]), dtype=torch.int64)}

perm = torch.randperm(170, generator=torch.Generator().manual_seed(SEED)).tolist()
ds_test = Subset(PennFudanDataset(DATA_DIR), perm[120:])            # the same 50 held-out images as notebook 01
TEST = [ds_test[i] for i in range(len(ds_test))]                    # preload into RAM (no disk I/O in timings)
BENCH_IMGS = [img for img, _ in TEST[:12]]
print(f"{len(TEST)} test images, benchmark subset {len(BENCH_IMGS)} images, sizes e.g. {tuple(BENCH_IMGS[0].shape)}")

def average_precision(preds, gts, iou_thr=0.5):
    n_gt = sum(len(g) for g in gts); S, T = [], []
    for p, g in zip(preds, gts):
        o = p["scores"].argsort(descending=True); boxes, scores = p["boxes"][o], p["scores"][o]
        matched = torch.zeros(len(g), dtype=torch.bool)
        ious = box_iou(boxes, g) if (len(boxes) and len(g)) else torch.zeros(len(boxes), len(g))
        for i in range(len(boxes)):
            tp = False
            if len(g):
                row = ious[i].clone(); row[matched] = -1.0; j = int(row.argmax())
                if row[j] >= iou_thr: matched[j] = True; tp = True
            S.append(float(scores[i])); T.append(tp)
    if not S: return 0.0
    o = np.argsort(-np.asarray(S), kind="stable"); tp = np.asarray(T, dtype=float)[o]
    tpc, fpc = np.cumsum(tp), np.cumsum(1 - tp)
    rec, prec = tpc / max(n_gt, 1), tpc / (tpc + fpc)
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[0.0], prec, [0.0]])
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

In [ ]:
WEIGHTS = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)

def load_detector():
    # Same architecture as notebook 01 (COCO weights are only used to build the graph; then overwritten).
    m = fasterrcnn_mobilenet_v3_large_fpn(weights=WEIGHTS)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 2)
    m.load_state_dict(ckpt["model"]); m.eval()
    return m

def configure(m, size=800, proposals=1000, detections=100):
    # Inference-time knobs - no retraining needed.
    m.transform.min_size = (size,)                       # short side after resize
    m.transform.max_size = int(round(size * 1333 / 800)) # keep the same aspect cap
    m.rpn._pre_nms_top_n["testing"] = proposals          # proposals kept per FPN level before NMS
    m.rpn._post_nms_top_n["testing"] = proposals         # proposals sent to the RoI box head
    m.roi_heads.detections_per_img = detections
    return m

def state_dict_mb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf); return buf.tell() / 1e6

def latency_ms(predict, repeats=2, trials=3, min_warmup_s=1.0):
    with torch.inference_mode():
        # Warm-up on EVERY benchmark image: each image has a different shape, and backends (oneDNN, ONNX Runtime)
        # prepare kernels per input shape on first use. Also warm up >= 1 s so the CPU clock has ramped up.
        t_end = time.perf_counter() + min_warmup_s
        for img in BENCH_IMGS: predict(img)
        while time.perf_counter() < t_end: predict(BENCH_IMGS[0])
        # Several trials, keep the best trial's median (like `timeit`): background activity on a laptop can only
        # make a trial slower, never faster, so the fastest trial is the most faithful estimate.
        medians = []
        for _ in range(trials):
            ts = []
            for _ in range(repeats):
                for img in BENCH_IMGS:
                    t0 = time.perf_counter(); predict(img); ts.append((time.perf_counter() - t0) * 1e3)
            medians.append(float(np.median(ts)))
    return min(medians)

def ap50(predict):
    with torch.inference_mode():
        preds = [predict(img) for img, _ in TEST]
    return average_precision(preds, [t["boxes"] for _, t in TEST])

results = []
def record(name, predict, size_mb, section, **info):
    lat = latency_ms(predict); ap = ap50(predict)
    row = {"section": section, "variant": name, "latency (ms)": round(lat, 1), "AP@0.5": round(ap, 3),
           "size (MB)": round(size_mb, 1), **info}
    results.append(row); print(row, flush=True)
    return row

torch_predict = lambda m: (lambda img: m([img])[0])

## (a) Baseline: FP32, PyTorch eager, 800 px, 1000 proposals, all threads

In [ ]:
model = configure(load_detector())
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f} M parameters, {state_dict_mb(model):.1f} MB")
base = record("baseline FP32 eager", torch_predict(model), state_dict_mb(model), "a", size_px=800, proposals=1000, threads=MAX_THREADS)

### Where does the time go? (profile one image)
Before optimising, **measure**. `torch.profiler` shows which operators dominate.

In [ ]:
from torch.profiler import profile, ProfilerActivity
with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU]) as prof:
    model([BENCH_IMGS[0]])
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=12))

# Rough parameter budget per component - where do the 19 M parameters live?
parts = {"backbone (MobileNetV3)": model.backbone.body, "FPN": model.backbone.fpn, "RPN": model.rpn,
         "box head (fc6+fc7)": model.roi_heads.box_head, "box predictor": model.roi_heads.box_predictor}
for k, v in parts.items():
    print(f"{k:24s} {sum(p.numel() for p in v.parameters())/1e6:6.2f} M params")

**Read the profile with the class:** convolutions (backbone+FPN) and the fully-connected box head (`addmm`/`linear`,
executed once *per proposal*) dominate. Note that the box head's `fc6` (12544 → 1024) alone holds ~12.8 M of the
19 M parameters — and it runs **1000 times per image** at the default setting. That observation drives sections (c) and (d).

## (b) Input resolution: the biggest knob for conv nets
Convolution cost scales with the number of pixels (≈ size²). Penn-Fudan people are large, so we may not need 800 px.
The model was trained multi-scale (480–800 px) in notebook 01, so it tolerates smaller inputs.

In [ ]:
for size in [640, 512, 416, 320]:
    configure(model, size=size)
    record(f"input {size}px", torch_predict(model), state_dict_mb(model), "b", size_px=size, proposals=1000, threads=MAX_THREADS)
configure(model)   # back to baseline

## (c) PyTorch dynamic INT8 quantization of `nn.Linear`

Dynamic quantization stores `nn.Linear` weights as INT8 and quantises activations on the fly. In this detector the only
Linear layers are in the **RoI box head** (fc6, fc7, predictor) — the convolutional backbone/FPN/RPN stay FP32.
So expect: **big size reduction** (fc6 is most of the parameters!) but only a **limited speed-up** (convs dominate).

> PyTorch 2.9 marks `torch.ao.quantization` as deprecated (to be removed in 2.10; successor: `torchao`).
> It still works in 2.9 — we silence the warning.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_q = torch.ao.quantization.quantize_dynamic(copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8)
print(model_q.roi_heads.box_head)
record("dynamic INT8 (Linear only)", torch_predict(model_q), state_dict_mb(model_q), "c", size_px=800, proposals=1000, threads=MAX_THREADS)

## (d) Fewer region proposals
The RPN keeps the top-N proposals after NMS and the box head processes **each of them** (RoIAlign + two FC layers).
Penn-Fudan images contain 1–8 people, so 1000 proposals is massive over-provisioning. (torchvision's 320-px mobile
variant uses 150 for the same reason.)

In [ ]:
for n in [300, 100, 50]:
    configure(model, proposals=n)
    record(f"{n} proposals", torch_predict(model), state_dict_mb(model), "d", size_px=800, proposals=n, threads=MAX_THREADS)
configure(model)

## (e) CPU threads
PyTorch uses intra-op parallelism (one op split across cores). More threads ≠ linear speed-up: small ops don't
parallelise well, and on hybrid laptop CPUs (performance + efficiency cores) extra threads can even slow things down.
In a server you often run **several worker processes with few threads each** for better throughput.
(Threads don't change the maths → AP is unchanged, so we only time it.)

In [ ]:
thread_rows = []
for t in sorted({1, 2, 4, 8, MAX_THREADS}):
    torch.set_num_threads(t)
    lat = latency_ms(torch_predict(model))
    thread_rows.append({"threads": t, "latency (ms)": round(lat, 1)}); print(thread_rows[-1], flush=True)
torch.set_num_threads(MAX_THREADS)
threads_df = pd.DataFrame(thread_rows)
best_t = int(threads_df.loc[threads_df["latency (ms)"].idxmin(), "threads"])
results.append({"section": "e", "variant": "1 thread", "latency (ms)": thread_rows[0]["latency (ms)"],
                "AP@0.5": base["AP@0.5"], "size (MB)": base["size (MB)"], "size_px": 800, "proposals": 1000, "threads": 1})
plt.figure(figsize=(5, 3.2)); plt.plot(threads_df["threads"], threads_df["latency (ms)"], "o-")
plt.xlabel("torch.set_num_threads"); plt.ylabel("latency per image (ms)"); plt.title("Threads vs latency"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "det_opt_threads.png", dpi=90); plt.show()
print("fastest thread count on this machine:", best_t)

## (f) ONNX export + ONNX Runtime

torchvision's detection models are written to be ONNX-exportable (the resize/normalise transform, RPN, RoIAlign and NMS
all have ONNX equivalents). We use the **TorchScript-based exporter** (`dynamo=False`, opset 17) — the classic path
that torchvision tests for detection models; the new `dynamo=True` exporter is the default in 2.9 but the data-dependent
control flow of detection post-processing is harder for it.

Notes:
* The resolution / proposal settings are **baked into the graph at export time** → export once per configuration.
* Dynamic axes for image height/width and for the number of detections.
* We check that ONNX Runtime reproduces the PyTorch outputs before trusting any timing.

In [ ]:
import onnx, onnxruntime as ort

def export_onnx(m, path):
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.onnx.export(m, ([BENCH_IMGS[0]],), str(path), opset_version=17, dynamo=False,
                          input_names=["image"], output_names=["boxes", "labels", "scores"],
                          dynamic_axes={"image": {1: "height", 2: "width"}, "boxes": {0: "n"},
                                        "labels": {0: "n"}, "scores": {0: "n"}})
    onnx.checker.check_model(onnx.load(str(path)))
    print(f"exported {path.name} in {time.perf_counter()-t0:.1f}s ({path.stat().st_size/1e6:.1f} MB)")

def ort_predictor(path):
    so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    def predict(img):
        b, l, s = sess.run(None, {"image": img.numpy()})
        return {"boxes": torch.from_numpy(b), "labels": torch.from_numpy(l), "scores": torch.from_numpy(s)}
    return predict

ONNX_BASE = CKPT_DIR / "fasterrcnn_pennfudan_800.onnx"
export_onnx(configure(model), ONNX_BASE)
onnx_pred = ort_predictor(ONNX_BASE)

# correctness check on a few images: same boxes & scores as PyTorch?
with torch.inference_mode():
    for img in BENCH_IMGS[:3]:
        a, b = model([img])[0], onnx_pred(img)
        k = min(len(a["scores"]), len(b["scores"]), 5)
        print(f"#dets torch={len(a['scores'])} onnx={len(b['scores'])} | top-{k} max |box diff| = "
              f"{(a['boxes'][:k] - b['boxes'][:k]).abs().max().item():.4f} px, max |score diff| = "
              f"{(a['scores'][:k] - b['scores'][:k]).abs().max().item():.5f}")
record("ONNX Runtime FP32", onnx_pred, ONNX_BASE.stat().st_size / 1e6, "f", size_px=800, proposals=1000, threads=MAX_THREADS)

## (g) Combine the knobs
Pick the settings that cost (almost) no accuracy in the sections above and stack them.

In [ ]:
COMBO_SIZE, COMBO_PROPOSALS = 512, 100
configure(model, size=COMBO_SIZE, proposals=COMBO_PROPOSALS)
record(f"{COMBO_SIZE}px + {COMBO_PROPOSALS} proposals (eager)", torch_predict(model), state_dict_mb(model), "g",
       size_px=COMBO_SIZE, proposals=COMBO_PROPOSALS, threads=MAX_THREADS)

model_q2 = configure(model_q, size=COMBO_SIZE, proposals=COMBO_PROPOSALS)
record(f"{COMBO_SIZE}px + {COMBO_PROPOSALS} proposals + dyn INT8", torch_predict(model_q2), state_dict_mb(model_q2), "g",
       size_px=COMBO_SIZE, proposals=COMBO_PROPOSALS, threads=MAX_THREADS)

ONNX_COMBO = CKPT_DIR / f"fasterrcnn_pennfudan_{COMBO_SIZE}_{COMBO_PROPOSALS}p.onnx"
export_onnx(model, ONNX_COMBO)
record(f"{COMBO_SIZE}px + {COMBO_PROPOSALS} proposals (ONNX Runtime)", ort_predictor(ONNX_COMBO), ONNX_COMBO.stat().st_size / 1e6, "g",
       size_px=COMBO_SIZE, proposals=COMBO_PROPOSALS, threads=MAX_THREADS)
configure(model)

## Final comparison

In [ ]:
df = pd.DataFrame(results)
b = df.iloc[0]
df["speed-up"] = (b["latency (ms)"] / df["latency (ms)"]).round(2)
df["ΔAP@0.5"] = (df["AP@0.5"] - b["AP@0.5"]).round(3)
df = df[["section", "variant", "size_px", "proposals", "threads", "size (MB)", "latency (ms)", "speed-up", "AP@0.5", "ΔAP@0.5"]]
display(df.style.hide(axis="index").format(precision=3)
          .background_gradient(subset=["speed-up"], cmap="Greens").background_gradient(subset=["ΔAP@0.5"], cmap="RdYlGn", vmin=-0.2, vmax=0.05))
df.to_csv(OUT_DIR / "det_optimization_results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
colors = {"a": "C3", "b": "C0", "c": "C1", "d": "C2", "e": "C7", "f": "C4", "g": "C5"}
for _, r in df.iterrows():
    axes[0].scatter(r["latency (ms)"], r["AP@0.5"], s=60, color=colors[r["section"]])
    axes[0].annotate(r["variant"], (r["latency (ms)"], r["AP@0.5"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
axes[0].set_xlabel("latency per image (ms, lower is better)"); axes[0].set_ylabel("AP@0.5 (higher is better)")
axes[0].set_title("Speed / accuracy trade-off (top-left is best)"); axes[0].grid(alpha=.3)
y = np.arange(len(df))
axes[1].barh(y, df["speed-up"], color=[colors[s] for s in df["section"]])
axes[1].set_yticks(y); axes[1].set_yticklabels(df["variant"], fontsize=8); axes[1].invert_yaxis()
axes[1].axvline(1, color="k", lw=.8); axes[1].set_xlabel("speed-up vs baseline (x)"); axes[1].set_title("Speed-up")
for yi, (s, a) in enumerate(zip(df["speed-up"], df["AP@0.5"])): axes[1].text(s, yi, f" {s:.2f}x  AP {a:.3f}", va="center", fontsize=7)
plt.tight_layout(); plt.savefig(OUT_DIR / "det_optimization_chart.png", dpi=90); plt.show()

### Visual sanity check: baseline vs fastest combination

In [ ]:
import matplotlib.patches as patches
fast = ort_predictor(ONNX_COMBO)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
with torch.inference_mode():
    for col, k in enumerate([1, 7, 13, 21]):
        img, t = TEST[k]
        for row, (pred, name) in enumerate([(torch_predict(model), "baseline 800px/1000p eager"),
                                            (fast, f"{COMBO_SIZE}px/{COMBO_PROPOSALS}p ONNX Runtime")]):
            p = pred(img); keep = p["scores"] >= 0.5
            ax = axes[row, col]; ax.imshow(img.permute(1, 2, 0)); ax.axis("off")
            for bx in t["boxes"].tolist():
                ax.add_patch(patches.Rectangle((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1], fill=False, ec="lime", lw=3))
            for bx, s in zip(p["boxes"][keep].tolist(), p["scores"][keep].tolist()):
                ax.add_patch(patches.Rectangle((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1], fill=False, ec="red", lw=1.5))
                ax.text(bx[0], bx[1] - 2, f"{s:.2f}", color="white", fontsize=7, bbox=dict(fc="red", alpha=.7, pad=1, ec="none"))
            ax.set_title(name, fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "det_opt_visual_check.png", dpi=80); plt.show()

## Take-aways: "optimisation is not only quantization"

1. **Profile first.** The cost is split between convolutions (scale with pixels) and the per-proposal box head
   (scales with the number of proposals). Each has its own knob.
2. **Input resolution** and **number of proposals** are free (no retraining) and typically give the largest speed-ups
   — but they *can* cost accuracy (small objects vanish at low resolution): always re-measure AP.
3. **Dynamic INT8 quantization** only touches `nn.Linear`: here it shrinks the file a lot (fc6 is huge) but speeds up
   inference only a little, because the convolutions stay FP32. Full INT8 of the conv backbone needs *static*
   quantization with calibration (e.g. ONNX Runtime / `torchao` PT2E / OpenVINO) — a bigger engineering step.
4. **Runtime matters:** ONNX Runtime executes the same maths with a different engine — measure whether it helps for
   *your* model and CPU (for small-ops models it usually does; for heavy conv models the gain can be modest).
5. **Threads:** more is not always better (hybrid CPUs, small ops); tune for the deployment box.
6. Always report **latency *and* accuracy** together — a 10× faster model that misses half the people is not optimised,
   it's broken.